# Experiment 1: Data Acquisition from Real‑World Sources


**Use Cases:** COVID‑19 / Weather / Stock Market Data Analysis

**Outcome:** Ability to collect, inspect, and understand real‑world datasets (raw vs clean)

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

## 2. Data Acquisition from CSV (COVID‑19 Data – Johns Hopkins)

In [ ]:
covid_url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
covid_df = pd.read_csv(covid_url)
print("COVID‑19 Dataset Shape:", covid_df.shape)
covid_df.head()

COVID‑19 Dataset Shape: (289, 1147)


,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


In [ ]:
covid_df.columns

Index(['Province/State', 'Country/Region', 'Lat', 'Long', '1/22/20', '1/23/20',
       '1/24/20', '1/25/20', '1/26/20', '1/27/20',
       ...
       '2/28/23', '3/1/23', '3/2/23', '3/3/23', '3/4/23', '3/5/23', '3/6/23',
       '3/7/23', '3/8/23', '3/9/23'],
      dtype='object', length=1147)

### Inspect Raw Data Issues

In [ ]:
covid_df.isnull().sum().head()

,0
Province/State,198
Country/Region,0
Lat,2
Long,2
1/22/20,0


### Simple Cleaning & Reshaping

In [ ]:
covid_long = covid_df.melt(
    id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
    var_name='Date',
    value_name='Confirmed_Cases'
)
covid_long['Date'] = pd.to_datetime(covid_long['Date'])
covid_long.head()

/tmp/ipython-input-472594675.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  covid_long['Date'] = pd.to_datetime(covid_long['Date'])


,Province/State,Country/Region,Lat,Long,Date,Confirmed_Cases
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0
1,NaN,Albania,41.15330,20.168300,2020-01-22,0
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0


## 3. Data Acquisition from Excel (Weather Dataset – Colab Upload)

In [ ]:
weather_df = pd.DataFrame({
    'Date': pd.date_range(start='2023-01-01', periods=5),
    'MinTemp': [20, 21, np.nan, 19, 22],
    'MaxTemp': [30, 32, 31, np.nan, 33],
    'Rainfall': [0.0, 5.2, 0.0, 1.1, np.nan]
})
weather_df

,Date,MinTemp,MaxTemp,Rainfall
0,2023-01-01,20.0,30.0,0.0
1,2023-01-02,21.0,32.0,5.2
2,2023-01-03,NaN,31.0,0.0
3,2023-01-04,19.0,NaN,1.1
4,2023-01-05,22.0,33.0,NaN


### Handling Missing Data

In [ ]:
weather_clean = weather_df.fillna(method='ffill')
weather_clean

/tmp/ipython-input-193554697.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  weather_clean = weather_df.fillna(method='ffill')


,Date,MinTemp,MaxTemp,Rainfall
0,2023-01-01,20.0,30.0,0.0
1,2023-01-02,21.0,32.0,5.2
2,2023-01-03,21.0,31.0,0.0
3,2023-01-04,19.0,31.0,1.1
4,2023-01-05,22.0,33.0,1.1


## 4. Data Acquisition from CSV (Stock Market Data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
stock_df = pd.DataFrame({
    'Date': pd.date_range(start='2023-01-01', periods=5),
    'Open': [150, 152, 151, 153, 154],
    'High': [155, 156, 154, 157, 158],
    'Low': [148, 150, 149, 151, 152],
    'Close': [154, 155, 153, 156, 157]
})
stock_df

,Date,Open,High,Low,Close
0,2023-01-01,150,155,148,154
1,2023-01-02,152,156,150,155
2,2023-01-03,151,154,149,153
3,2023-01-04,153,157,151,156
4,2023-01-05,154,158,152,157


## 5. Web Scraping using BeautifulSoup

In [ ]:
url = "https://www.worldometers.info/coronavirus/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
covid_table = soup.find('table', id='main_table_countries_today')
headers = [th.text.strip() for th in covid_table.find_all('th')]
rows = []
for tr in covid_table.find_all('tr')[1:11]:
    cells = [td.text.strip() for td in tr.find_all('td')]
    rows.append(cells)
web_covid_df = pd.DataFrame(rows, columns=headers[:len(rows[0])])
web_covid_df.head()

,#,"Country,Other",TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,"Serious,Critical",...,TotalTests,Tests/\n1M pop,Population,Continent,1 Caseevery X ppl,1 Deathevery X ppl,1 Testevery X ppl,New Cases/1M pop,New Deaths/1M pop,Active Cases/1M pop
0,,North America,"131,889,132",,"1,695,941",,"127,665,129",+350,"2,528,062","6,095",...,,,,North America,,,,,,
1,,Asia,"221,500,265",,"1,553,662",,"205,673,091",,"14,273,512","14,733",...,,,,Asia,,,,,,
2,,Europe,"253,406,198",,"2,101,824",,"248,754,104",+474,"2,550,270","4,453",...,,,,Europe,,,,,,
3,,South America,"70,200,879",,"1,367,332",,"66,683,585",,"2,149,962","8,953",...,,,,South America,,,,,,
4,,Oceania,"14,895,771",,"33,015",,"14,752,388",,"110,368",31,...,,,,Australia/Oceania,,,,,,


## 6. Raw vs Clean Data

In [ ]:
clean_web_covid = web_covid_df.replace('', np.nan)
clean_web_covid.head()

/tmp/ipython-input-1014330463.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean_web_covid = web_covid_df.replace('', np.nan)


,#,"Country,Other",TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,"Serious,Critical",...,TotalTests,Tests/\n1M pop,Population,Continent,1 Caseevery X ppl,1 Deathevery X ppl,1 Testevery X ppl,New Cases/1M pop,New Deaths/1M pop,Active Cases/1M pop
0,NaN,North America,"131,889,132",NaN,"1,695,941",NaN,"127,665,129",+350,"2,528,062","6,095",...,NaN,NaN,NaN,North America,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Asia,"221,500,265",NaN,"1,553,662",NaN,"205,673,091",NaN,"14,273,512","14,733",...,NaN,NaN,NaN,Asia,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Europe,"253,406,198",NaN,"2,101,824",NaN,"248,754,104",+474,"2,550,270","4,453",...,NaN,NaN,NaN,Europe,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,South America,"70,200,879",NaN,"1,367,332",NaN,"66,683,585",NaN,"2,149,962","8,953",...,NaN,NaN,NaN,South America,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Oceania,"14,895,771",NaN,"33,015",NaN,"14,752,388",NaN,"110,368",31,...,NaN,NaN,NaN,Australia/Oceania,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Conclusion
- Data loaded from CSV, Excel, and Web
- Raw vs clean data understood
- Web scraping performed using BeautifulSoup

**Learning Outcome Achieved**